<a href="https://colab.research.google.com/github/MarwahAlaofi/acm-proceedings/blob/main/OpenReview_to_ACM_XML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openreview-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 33.6 MB/s eta 0:00:00
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=6ad2d660d396e5b2fd421806b1661aa2036f0db74761a42acefca28c5b11f93d
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
import os
from google.colab import userdata

try:
    # Fetching credentials from Colab Secrets (the 🔑 icon on the left)
    # These variables will now be available in the kernel for the next cell
    os.environ['OPENREVIEW_USERNAME'] = userdata.get('OPENREVIEW_USERNAME')
    os.environ['OPENREVIEW_PASSWORD'] = userdata.get('OPENREVIEW_PASSWORD')
    print('Credentials loaded successfully from Colab Secrets.')
except userdata.SecretNotFoundError:
    print('ERROR: Please add OPENREVIEW_USERNAME and OPENREVIEW_PASSWORD to your Colab Secrets (🔑 sidebar).')

Credentials loaded successfully from Colab Secrets.


In [6]:
import openreview
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
import os
import sys
import logging
from tqdm import tqdm

# Configure logging
# Added force=True to override Colab's default hidden logger settings
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%H:%M:%S',
    stream=sys.stdout,
    force=True
)

# ----------------------------
# Format/Data Helpers
# ----------------------------
def format_date(timestamp):
    """
    Converts an OpenReview timestamp (in milliseconds) to an ACM-compliant
    date string (e.g., 'DD-MMM-YYYY').
    """
    if not timestamp:
        return ""
    return datetime.fromtimestamp(timestamp / 1000, timezone.utc).strftime("%d-%b-%Y").upper()

def indent(elem, level=0):
    """
    Recursively adds indentation and newlines to an XML ElementTree,
    enabling pretty-printing of the final XML output file.
    """
    i = "\n" + level * "  "
    if len(elem):
        if not elem.text or not elem.text.strip():
            elem.text = i + "  "
        for child in elem:
            indent(child, level + 1)
        if not child.tail or not child.tail.strip():
            child.tail = i
    if level and (not elem.tail or not elem.tail.strip()):
        elem.tail = i

def get_submission_date(submission):
    """
    Extracts the submission creation date from an OpenReview note object.
    OpenReview uses 'tcdate' (true creation date) or 'cdate' for timestamps.
    """
    timestamp = getattr(submission, "tcdate", None) or getattr(submission, "cdate", None)
    return format_date(timestamp)

def get_preferred_email(profile):
    """Safely extracts the preferred email from a profile."""
    try:
        return profile.get_preferred_email()
    except Exception:
        return ""

def get_current_affiliations(profile, author_id=""):
    """
    Extracts all *current* affiliations from a profile.
    An affiliation is considered current if it has no 'end' date.
    Deduplicates identical institutions to prevent redundant XML blocks.
    """
    affiliations = []
    default_aff = {"department": "", "institution": "", "city": "", "state": "", "country": ""}

    try:
        if not profile:
            return [default_aff]

        content = getattr(profile, 'content', {})
        history = content.get("history", [])

        if not history:
            return [default_aff]

        # Filter for current affiliations (no endDate)
        current_history = [h for h in history if not h.get("endDate")]

        # If no active found, fallback to the most recent one
        if not current_history:
            current_history = [history[0]]

        seen_institutions = set()

        for entry in current_history:
            inst = entry.get("institution", {})
            if isinstance(inst, str):
                inst_name = inst
                dept = ""
                city = ""
                state_prov = ""
                country = ""
            else:
                inst_name = inst.get("name") or ""
                dept = inst.get("department") or ""
                city = inst.get("city") or ""
                state_prov = inst.get("stateProvince") or ""
                country = inst.get("country") or ""

            # Create a unique key (institution + department) for deduplication
            dedup_key = (inst_name.lower().strip(), dept.lower().strip())

            if dedup_key not in seen_institutions and (inst_name or dept):
                seen_institutions.add(dedup_key)
                affiliations.append({
                    "department": dept,
                    "institution": inst_name,
                    "city": city,
                    "state": state_prov,
                    "country": country
                })

        return affiliations if affiliations else [default_aff]

    except Exception as e:
        logging.error(f"Exception parsing affiliation for {author_id}: {e}")
        return [default_aff]

def get_author_name(name_string, profile):
    """
    Extracts the preferred structured name (first, middle, last) from an
    OpenReview profile to ensure accurate metadata.
    Falls back to a basic string split if the author has no OR profile OR
    if the profile name record is empty.
    """
    first, middle, last = "", "", ""

    if profile:
        names = profile.content.get("names", [])
        for n in names:
            if n.get("preferred"):
                first = n.get("first") or ""
                middle = n.get("middle") or ""
                last = n.get("last") or ""
                break

        # If preferred name is totally empty, fallback to the first listed name
        if not first and not last and names:
            first = names[0].get("first") or ""
            middle = names[0].get("middle") or ""
            last = names[0].get("last") or ""

    # Ultimate Fallback: If no profile exists OR the profile yielded empty strings,
    # we split the raw name string provided in the paper submission.
    if not first and not last and name_string:
        parts = name_string.strip().split()
        if len(parts) == 1:
            first = parts[0]
        elif len(parts) >= 2:
            first = parts[0]
            last = parts[-1]
            if len(parts) > 2:
                middle = " ".join(parts[1:-1])

    return first.strip(), middle.strip(), last.strip()

def get_profiles_map(client, author_ids):
    """
    Batch fetches OpenReview profiles to avoid API rate limits and speed up execution.
    Filters for valid OpenReview tilde IDs (e.g., ~John_Doe1) and returns a map
    of ID strings to Profile objects.
    """
    tilde_ids = [aid for aid in author_ids if aid.startswith("~")]
    id2profile = {}

    if tilde_ids:
        try:
            logging.info(f"Fetching {len(tilde_ids)} profiles from OpenReview...")
            profiles = openreview.tools.get_profiles(client, tilde_ids, with_publications=False)
            for p in profiles:
                id2profile[p.id] = p
                # Also map associated emails to the profile
                for email in p.content.get("emails", []):
                    id2profile[email] = p

            logging.info(f"Successfully fetched {len(profiles)} unique profiles.")

            # Identify missing
            missing_count = len(tilde_ids) - len(profiles)
            if missing_count > 0:
                logging.warning(f"Could not find valid profiles for {missing_count} requested tilde IDs.")

        except Exception as e:
            logging.error(f"Error fetching profiles: {e}")

    return id2profile

# ----------------------------
# Main Export Function
# ----------------------------
def export_acm_xml(venue_id, paper_type, output_file="acm_output.xml"):
    """
    Main orchestration function. Connects to the OpenReview API, fetches
    accepted papers for a specific venue, resolves author metadata, and
    compiles it into the ACM/Sheridan XML format.
    """
    # Initialize the OpenReview API V2 client using environment variables
    client = openreview.api.OpenReviewClient(
        baseurl="https://api2.openreview.net",
        username=os.getenv('OPENREVIEW_USERNAME'),
        password=os.getenv('OPENREVIEW_PASSWORD')
    )

    # Tracking Statistics for the final summary
    stats = {
        "total_papers": 0,
        "total_authors": 0,
        "missing_profiles": 0,
        "missing_first_names": 0,
        "missing_last_names": 0,
        "missing_emails": 0,
        "missing_institutions": 0,
        "missing_departments": 0,
        "missing_cities": 0,
        "missing_states": 0,
        "missing_countries": 0
    }

    try:
        logging.info(f"Fetching submissions for {venue_id}...")
        submissions = client.get_all_notes(content={"venueid": venue_id})
        if not submissions:
            logging.warning("No submissions found.")
            return

        stats["total_papers"] = len(submissions)
        logging.info(f"Found {stats['total_papers']} submissions.")
    except Exception as e:
        logging.error(f"CRITICAL ERROR: {e}")
        return

    logging.info("Collecting author IDs...")
    # API V2 wraps field data inside a 'value' key.
    author_ids = {aid for s in submissions for aid in s.content.get("authorids", {}).get("value", []) if isinstance(aid, str)}

    logging.info(f"Found {len(author_ids)} unique author IDs across all papers.")
    # Batch load all profiles to minimize API calls during XML generation
    id2profile = get_profiles_map(client, list(author_ids))

    logging.info("Building XML tree...")
    # Root element for ACM digital library schema
    root = ET.Element("erights_record")
    parent = ET.SubElement(root, "parent_data")
    ET.SubElement(parent, "proceeding").text = venue_id
    ET.SubElement(parent, "volume").text = ""
    ET.SubElement(parent, "issue").text = ""
    ET.SubElement(parent, "issue_date").text = ""
    ET.SubElement(parent, "source").text = "OpenReview"

    # Iteratively build the XML tree for each paper
    for seq, s in enumerate(tqdm(submissions, desc="Processing papers"), 1):
        paper = ET.SubElement(root, "paper")

        # Paper fields
        title = s.content.get("title", {}).get("value", "")
        ET.SubElement(paper, "paper_type").text = paper_type
        ET.SubElement(paper, "art_submission_date").text = get_submission_date(s)

        # Add art_approval_date to match Sheridan reference
        approval_ts = getattr(s, "tmdate", None) or getattr(s, "cdate", None)
        ET.SubElement(paper, "art_approval_date").text = format_date(approval_ts)

        ET.SubElement(paper, "paper_title").text = title
        ET.SubElement(paper, "event_tracking_number").text = s.id
        ET.SubElement(paper, "published_article_number").text = ""
        ET.SubElement(paper, "start_page").text = ""
        ET.SubElement(paper, "end_page").text = ""

        authors_xml = ET.SubElement(paper, "authors")
        names = s.content.get("authors", {}).get("value", [])
        ids = s.content.get("authorids", {}).get("value", [])

        for i, (name_str, aid) in enumerate(zip(names, ids), start=1):
            stats["total_authors"] += 1
            author_xml = ET.SubElement(authors_xml, "author")
            profile = id2profile.get(aid)

            if not profile:
                stats["missing_profiles"] += 1

            first, middle, last = get_author_name(name_str, profile)

            if not first: stats["missing_first_names"] += 1
            if not last: stats["missing_last_names"] += 1

            ET.SubElement(author_xml, "prefix").text = ""
            ET.SubElement(author_xml, "first_name").text = first
            ET.SubElement(author_xml, "middle_name").text = middle
            ET.SubElement(author_xml, "last_name").text = last
            ET.SubElement(author_xml, "suffix").text = ""

            # Affiliations
            affs_xml = ET.SubElement(author_xml, "affiliations")
            affiliations = get_current_affiliations(profile, aid)

            has_institution = False
            has_department = False
            has_city = False
            has_state = False
            has_country = False

            for seq_no, aff in enumerate(affiliations, start=1):
                if aff.get("institution"): has_institution = True
                if aff.get("department"): has_department = True
                if aff.get("city"): has_city = True
                if aff.get("state"): has_state = True
                if aff.get("country"): has_country = True

                aff_xml = ET.SubElement(affs_xml, "affiliation")
                ET.SubElement(aff_xml, "department").text = aff.get("department", "")
                ET.SubElement(aff_xml, "institution").text = aff.get("institution", "")
                ET.SubElement(aff_xml, "city").text = aff.get("city", "")
                ET.SubElement(aff_xml, "state_province").text = aff.get("state", "")
                ET.SubElement(aff_xml, "country").text = aff.get("country", "")
                ET.SubElement(aff_xml, "sequence_no").text = str(seq_no)

            if not has_institution: stats["missing_institutions"] += 1
            if not has_department: stats["missing_departments"] += 1
            if not has_city: stats["missing_cities"] += 1
            if not has_state: stats["missing_states"] += 1
            if not has_country: stats["missing_countries"] += 1

            # Email
            email = get_preferred_email(profile) if profile else (aid if "@" in aid else "")
            if not email:
                stats["missing_emails"] += 1

            ET.SubElement(author_xml, "email_address").text = email
            ET.SubElement(author_xml, "sequence_no").text = str(i)
            ET.SubElement(author_xml, "contact_author").text = "Y" if i == 1 else "N"
            ET.SubElement(author_xml, "ACM_profile_id").text = ""
            ET.SubElement(author_xml, "ACM_client_no").text = ""
            ET.SubElement(author_xml, "ORCID").text = ""
            ET.SubElement(author_xml, "role").text = "author"

        # sequence_no moved after page tags and authors block per schema
        ET.SubElement(paper, "sequence_no").text = str(seq)
        ET.SubElement(paper, "section").text = ""

    # Prettify the XML string before writing
    indent(root)
    ET.ElementTree(root).write(output_file, encoding="utf-8", xml_declaration=True)


    # ========================================================================
    # VALIDATE OUTPUT
    # ========================================================================
    logging.info("\n" + "="*40)
    logging.info("         VALIDATING OUTPUT              ")
    logging.info("="*40)

    validation_passed = True
    issues = []

    # Validate exactly one contact author per paper
    for paper_elem in root.findall("paper"):
        paper_id = paper_elem.findtext("event_tracking_number", "unknown")
        authors = paper_elem.findall(".//author")
        contact_authors = [a for a in authors if a.findtext("contact_author") == "Y"]

        if len(contact_authors) == 0:
            issues.append(f"Paper {paper_id}: No contact author")
            validation_passed = False
        elif len(contact_authors) > 1:
            issues.append(f"Paper {paper_id}: Multiple contact authors ({len(contact_authors)})")
            validation_passed = False

    if validation_passed:
        logging.info(" ✓ All papers have exactly one contact author")
    else:
        logging.error(f" ✗ Validation FAILED: {len(issues)} issue(s) found")
        for issue in issues[:10]:
            logging.error(f"   {issue}")
        if len(issues) > 10:
            logging.error(f"   ... and {len(issues) - 10} more")
        logging.warning(" WARNING: Output may not meet ACM requirements")
    logging.info("="*40)

    # Output Detailed Summary
    logging.info("\n" + "="*40)
    logging.info("         DATA EXTRACTION SUMMARY        ")
    logging.info("="*40)
    logging.info(f" Papers Processed         : {stats['total_papers']}")
    logging.info(f" Authors Processed        : {stats['total_authors']}")
    logging.info("-" * 40)
    logging.info(f" Missing OR Profiles      : {stats['missing_profiles']}")
    logging.info(f" Missing First Names      : {stats['missing_first_names']}")
    logging.info(f" Missing Last Names       : {stats['missing_last_names']}")
    logging.info(f" Missing Email Addresses  : {stats['missing_emails']}")
    logging.info(f" Missing Institutions     : {stats['missing_institutions']}")
    logging.info(f" Missing Departments      : {stats['missing_departments']}")
    logging.info(f" Missing Cities           : {stats['missing_cities']}")
    logging.info(f" Missing States/Provinces : {stats['missing_states']}")
    logging.info(f" Missing Countries        : {stats['missing_countries']}")
    logging.info("="*40)
    logging.info(f" Done! XML generated at: {output_file}\n")

if __name__ == "__main__":

    VENUE_ID = "ICLR.cc/2024/Conference"
    PAPER_TYPE = "Full Paper"
    OUTPUT_FILE = "full_papers_export.xml"

    export_acm_xml(venue_id=VENUE_ID, paper_type=PAPER_TYPE, output_file=OUTPUT_FILE)

13:55:47 - INFO - Fetching submissions for ICLR.cc/2024/Conference...
13:55:47 - INFO - Found 2260 submissions.
13:55:47 - INFO - Collecting author IDs...
13:55:47 - INFO - Found 9257 unique author IDs across all papers.
13:55:47 - INFO - Fetching 9195 profiles from OpenReview...
13:55:54 - INFO - Successfully fetched 9182 unique profiles.
13:55:54 - WARNING - Could not find valid profiles for 13 requested tilde IDs.
13:55:54 - INFO - Building XML tree...


Processing papers: 100%|██████████| 2260/2260 [00:01<00:00, 1485.56it/s]


13:55:57 - INFO - 
13:55:57 - INFO -          DATA EXTRACTION SUMMARY        
13:55:57 - INFO - ========================================
13:55:57 - INFO -  Papers Processed         : 2260
13:55:57 - INFO -  Authors Processed        : 11999
13:55:57 - INFO - ----------------------------------------
13:55:57 - INFO -  Missing OR Profiles      : 912
13:55:57 - INFO -  Missing First Names      : 0
13:55:57 - INFO -  Missing Last Names       : 1
13:55:57 - INFO -  Missing Email Addresses  : 849
13:55:57 - INFO -  Missing Institutions     : 913
13:55:57 - INFO -  Missing Departments      : 7829
13:55:57 - INFO -  Missing Cities           : 6528
13:55:57 - INFO -  Missing States/Provinces : 6792
13:55:57 - INFO -  Missing Countries        : 2863
13:55:57 - INFO - ========================================
13:55:57 - INFO -  Done! XML generated at: full_papers_export.xml

